# Maximum Likelihood Estimation
**Topic:** Probability & Statistics

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go

import ipywidgets as widgets
from ipywidgets import FloatSlider, Output, VBox

from IPython.display import display, clear_output
from scipy import stats

from tkh_utils import PALETTE, FONT, base_layout
from tkh_utils import make_output, make_slider, display_widget, register_observer

---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** the difference between probability and likelihood
- **Explain** maximum likelihood estimation as the process of finding parameters that make observed data most probable
- **Interpret** how MLE connects to what `model.fit()` does under the hood in scikit-learn

> **Tip:** Drag the sliders to try to maximize the log-likelihood score displayed in the chart title. Notice where the score peaks, then compare that to the dotted MLE line the computer found automatically.

---
## How we got here

In *12: The Normal Distribution* we learned that a normal distribution is defined by two parameters, μ and σ. And in *14: Sampling Distributions* we saw that different samples produce different estimates of those parameters. A natural question follows: given a set of data points, which values of μ and σ fit the data best? Maximum likelihood estimation is the standard answer to that question.

---
## Why this matters for data science

Every time you call `model.fit(X, y)` in scikit-learn, something is being optimized. For logistic regression that something is log-likelihood. For naive Bayes it is likelihood directly. For linear regression it is equivalent to maximizing likelihood under a Gaussian noise assumption. MLE is the mathematical engine behind most parametric machine learning models, and understanding it makes model training go from magic to mechanism.

---
## Try it yourself

In [2]:
np.random.seed(42)
DATA = np.random.normal(loc=5.2, scale=1.4, size=60)
MLE_MU = np.mean(DATA)
MLE_SIGMA = np.std(DATA)
MLE_LL = np.sum(stats.norm.logpdf(DATA, MLE_MU, MLE_SIGMA))

x_range = np.linspace(0.5, 10.0, 400)

out = make_output()
mu_slider    = make_slider(1.0, 9.0, 0.1, 3.5, "Mean (μ):")
sigma_slider = make_slider(0.3, 3.5, 0.1, 2.5, "Std dev (σ):")


def render(mu, sigma):
    with out:
        clear_output(wait=True)

        ll = np.sum(stats.norm.logpdf(DATA, mu, sigma))
        pct_of_max = (ll / MLE_LL) * 100 if MLE_LL != 0 else 0

        traces = [
            go.Histogram(
                x=DATA,
                histnorm="probability density",
                nbinsx=16,
                marker_color=PALETTE["primary"],
                opacity=0.35,
                name="Observed data",
            ),
            go.Scatter(
                x=x_range,
                y=stats.norm.pdf(x_range, mu, sigma),
                mode="lines",
                line=dict(color=PALETTE["secondary"], width=3),
                name=f"Your fit: N({mu:.1f}, {sigma:.1f})",
            ),
            go.Scatter(
                x=x_range,
                y=stats.norm.pdf(x_range, MLE_MU, MLE_SIGMA),
                mode="lines",
                line=dict(color=PALETTE["accent"], width=2, dash="dot"),
                name=f"MLE solution: N({MLE_MU:.2f}, {MLE_SIGMA:.2f})",
            ),
        ]

        layout = base_layout(
            title=f"Log-likelihood: {ll:.1f}   (MLE maximum: {MLE_LL:.1f})   You are at {pct_of_max:.0f}% of optimal",
            xaxis_title="Value",
            yaxis_title="Probability Density",
        )
        layout.update(showlegend=True)
        fig = go.Figure(data=traces, layout=layout)
        display(go.FigureWidget(fig))


register_observer([mu_slider, sigma_slider],
                  lambda change: render(mu_slider.value, sigma_slider.value))

display_widget(VBox([mu_slider, sigma_slider]), out)
render(mu_slider.value, sigma_slider.value)

---
## What's happening?

**Probability vs likelihood** sounds like a subtle distinction but it matters:

- **Probability** asks: given fixed parameters, how probable is a particular outcome?
- **Likelihood** asks: given fixed data, how plausible are a particular set of parameters?

For a single data point $x$ drawn from $N(\mu, \sigma)$, the likelihood is just the density at that point:

$$\mathcal{L}(\mu, \sigma \mid x) = f(x; \mu, \sigma) = \frac{1}{\sigma\sqrt{2\pi}} \, e^{-\frac{(x-\mu)^2}{2\sigma^2}}$$

For $n$ independent data points, the joint likelihood is the product of all individual densities:

$$\mathcal{L}(\mu, \sigma \mid x_1, \ldots, x_n) = \prod_{i=1}^{n} f(x_i; \mu, \sigma)$$

Because multiplying many small fractions quickly becomes numerically unstable, we take the log. Since log is monotonic, **maximizing the log-likelihood finds the same parameters** as maximizing the likelihood itself:

$$\ell(\mu, \sigma) = \sum_{i=1}^{n} \log f(x_i; \mu, \sigma)$$

### The MLE solution for a normal distribution

Taking derivatives and setting them to zero gives a closed-form result:

| Parameter | MLE estimator |
|-----------|---------------|
| μ | $\hat{\mu} = \bar{x}$ (sample mean) |
| σ | $\hat{\sigma} = \sqrt{\frac{1}{n}\sum(x_i - \bar{x})^2}$ (population std dev) |

The sample mean is not just convenient, it is the mathematically optimal estimator under the assumption that data came from a normal distribution.

---
## Real-world example: Fitting a model to delivery times

A logistics company records 200 delivery times. They want to model this distribution so they can quote customers a reliable delivery window. The chart below shows how the MLE fit compares to a naive guess and how the log-likelihood score reflects quality of fit.

Notice:

- The MLE fit (green) sits nearly on top of the histogram even though it never "sees" the bins, only the raw data points
- The naive guess (orange) misses the center and has a much lower log-likelihood score
- The gap between the scores reflects how much worse the naive model would be at predicting new deliveries

> **Discussion question:** A delivery time cannot be negative, but a normal distribution assigns some probability to negative values. What does this tell you about the assumption built into MLE when you choose a normal model?

In [3]:
np.random.seed(7)
delivery_times = np.random.normal(loc=42, scale=8, size=200).clip(min=10)

mle_mu    = np.mean(delivery_times)
mle_sigma = np.std(delivery_times)
guess_mu, guess_sigma = 50.0, 15.0

ll_mle   = np.sum(stats.norm.logpdf(delivery_times, mle_mu, mle_sigma))
ll_guess = np.sum(stats.norm.logpdf(delivery_times, guess_mu, guess_sigma))

x_d = np.linspace(5, 80, 400)

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=delivery_times, histnorm="probability density", nbinsx=25,
    marker_color=PALETTE["primary"], opacity=0.4, name="Delivery times (n=200)"
))
fig.add_trace(go.Scatter(
    x=x_d, y=stats.norm.pdf(x_d, mle_mu, mle_sigma),
    mode="lines", line=dict(color=PALETTE["accent"], width=3),
    name=f"MLE fit  μ={mle_mu:.1f}, σ={mle_sigma:.1f}  (ℓ={ll_mle:.0f})"
))
fig.add_trace(go.Scatter(
    x=x_d, y=stats.norm.pdf(x_d, guess_mu, guess_sigma),
    mode="lines", line=dict(color=PALETTE["secondary"], width=2.5, dash="dash"),
    name=f"Naive guess  μ={guess_mu}, σ={guess_sigma}  (ℓ={ll_guess:.0f})"
))

layout = base_layout(
    title="MLE Fit vs Naive Guess — Package Delivery Times",
    xaxis_title="Delivery time (minutes)",
    yaxis_title="Probability Density",
)
layout.update(showlegend=True)
fig.update_layout(layout)
fig.show()

### MLE in other common models

| Model | What MLE optimizes | Loss function it corresponds to |
|-------|-------------------|----------------------------------|
| Linear regression | Gaussian likelihood on residuals | Mean squared error |
| Logistic regression | Bernoulli likelihood on class labels | Binary cross-entropy |
| Naive Bayes | Product of class-conditional likelihoods | Log-loss |
| Neural network classifier | Categorical likelihood on labels | Categorical cross-entropy |
| Poisson regression | Poisson likelihood on count data | Poisson deviance |

---
## Key takeaway

> **Maximum likelihood estimation finds the parameter values that make the observed data most probable under a chosen model, and this is exactly the optimization that `model.fit()` performs.**

---
*Next up: Bootstrap & Resampling — how to estimate uncertainty around any statistic without assuming a distribution*